In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload vent_dataset.zip to your Drive root first, then:
import zipfile, os

zip_path = '/content/drive/MyDrive/Datasets/vent_dataset.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')

print("Files:", os.listdir('/content/'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.load('/content/vent_X_train.npy')
Y = np.load('/content/vent_Y_train.npy')

print("=== BASIC STATS ===")
print(f"X shape: {X.shape}  |  Y shape: {Y.shape}")
print(f"X dtype: {X.dtype}  |  Y dtype: {Y.dtype}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")
print(f"X mean:  {X.mean():.3f}  |  X std: {X.std():.3f}")
print(f"\nClass balance:")
print(f"  Safe   (0): {(Y==0).sum():>8,}  ({(Y==0).mean()*100:.1f}%)")
print(f"  Crisis (1): {(Y==1).sum():>8,}  ({(Y==1).mean()*100:.1f}%)")
print(f"\nNaN in X: {np.isnan(X).sum()}")
print(f"NaN in Y: {np.isnan(Y).sum()}")
print(f"Negative values in X: {(X < 0).sum()}")

In [ ]:
safe_idx   = np.where(Y == 0)[0]
crisis_idx = np.where(Y == 1)[0]

# Compute mean waveform per class
safe_mean   = X[safe_idx, 0, :].mean(axis=0)
crisis_mean = X[crisis_idx, 0, :].mean(axis=0)
safe_std    = X[safe_idx, 0, :].std(axis=0)
crisis_std  = X[crisis_idx, 0, :].std(axis=0)

t = np.arange(100) * 0.01

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].fill_between(t, safe_mean - safe_std,   safe_mean + safe_std,   alpha=0.2, color='green')
axes[0].fill_between(t, crisis_mean - crisis_std, crisis_mean + crisis_std, alpha=0.2, color='red')
axes[0].plot(t, safe_mean,   color='green', label='Safe mean',   linewidth=2)
axes[0].plot(t, crisis_mean, color='red',   label='Crisis mean', linewidth=2)
axes[0].set_title('Mean Waveform per Class (CRITICAL CHECK)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('CO2 (mmHg)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Difference plot — should show a clear signal, not noise
diff = crisis_mean - safe_mean
axes[1].bar(t, diff, width=0.009, color=['red' if d > 0 else 'green' for d in diff])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Crisis minus Safe (should NOT be near-zero flat line)')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('CO2 difference (mmHg)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/audit_class_separation.png', dpi=150)
plt.show()

print(f"\nMean CO2 — Safe: {safe_mean.mean():.2f}  Crisis: {crisis_mean.mean():.2f}")
print(f"Max absolute difference between class means: {np.abs(diff).max():.3f} mmHg")
print(f"If this is < 2.0 mmHg the labeling logic is likely broken.")

In [ ]:
# Sample 20 crisis and 20 safe chunks and print their stats
np.random.seed(42)
sample_crisis = np.random.choice(crisis_idx, min(20, len(crisis_idx)), replace=False)
sample_safe   = np.random.choice(safe_idx,   min(20, len(safe_idx)),   replace=False)

print("=== CRISIS CHUNK STATS (CO2 should be near 0 or near/above 55) ===")
for i, idx in enumerate(sample_crisis[:10]):
    chunk = X[idx, 0, :]
    print(f"  [{i:>2}] min={chunk.min():.1f}  max={chunk.max():.1f}  "
          f"mean={chunk.mean():.1f}  std={chunk.std():.2f}")

print("\n=== SAFE CHUNK STATS (CO2 should be 15–55 mmHg, rhythmic) ===")
for i, idx in enumerate(sample_safe[:10]):
    chunk = X[idx, 0, :]
    print(f"  [{i:>2}] min={chunk.min():.1f}  max={chunk.max():.1f}  "
          f"mean={chunk.mean():.1f}  std={chunk.std():.2f}")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
axes = axes.flatten()

# First 8 = safe, last 8 = crisis
for col, idx in enumerate(sample_safe[:8]):
    axes[col].plot(t, X[idx, 0, :], color='green', linewidth=1.2)
    axes[col].set_title(f'Safe #{col}', fontsize=9)
    axes[col].set_ylim(-5, 70)
    axes[col].grid(alpha=0.3)

for col, idx in enumerate(sample_crisis[:8]):
    axes[col+8].plot(t, X[idx, 0, :], color='red', linewidth=1.2)
    axes[col+8].set_title(f'Crisis #{col}', fontsize=9)
    axes[col+8].set_ylim(-5, 70)
    axes[col+8].grid(alpha=0.3)

plt.suptitle('Top row: Safe  |  Bottom row: Crisis  (visually inspect these)', fontsize=12)
plt.tight_layout()
plt.savefig('/content/audit_individual_waveforms.png', dpi=150)
plt.show()

In [ ]:
# Find transitions: rows where label flips from 0 → 1
# Then look at the actual waveform 5 minutes BEFORE that transition
# and the waveform AT the transition — they should look very different

transitions = np.where(np.diff(Y) == 1)[0]   # indices just before a 0→1 flip
print(f"Found {len(transitions)} Safe→Crisis transitions in the dataset")

if len(transitions) > 0:
    fig, axes = plt.subplots(min(4, len(transitions)), 3, figsize=(15, 10))
    if len(transitions) == 1:
        axes = axes[np.newaxis, :]

    for row, t_idx in enumerate(transitions[:4]):
        # Waveform well before the crisis zone
        before_idx = max(0, t_idx - 50)
        at_idx     = t_idx
        after_idx  = min(len(Y) - 1, t_idx + 50)

        for col, (chunk_idx, label) in enumerate([
            (before_idx, Y[before_idx]),
            (at_idx,     Y[at_idx]),
            (after_idx,  Y[after_idx])
        ]):
            chunk = X[chunk_idx, 0, :]
            color = 'red' if label == 1 else 'green'
            axes[row, col].plot(np.arange(100)*0.01, chunk, color=color, linewidth=1.2)
            axes[row, col].set_title(
                f'Transition {row+1} | {"50 before" if col==0 else "at boundary" if col==1 else "50 after"} '
                f'| Label={int(label)}', fontsize=8)
            axes[row, col].set_ylim(-5, 70)
            axes[row, col].grid(alpha=0.3)

    plt.suptitle('Label Boundary Inspection\nGreen=Safe, Red=Crisis — waveforms should DIFFER between rows', fontsize=11)
    plt.tight_layout()
    plt.savefig('/content/audit_transitions.png', dpi=150)
    plt.show()

In [ ]:
print("=== NORMALIZATION AUDIT ===")
print(f"Global X range: [{X.min():.2f}, {X.max():.2f}]")
print(f"Global X mean:  {X.mean():.2f}  std: {X.std():.2f}")

# Per-chunk stats — flag outliers
chunk_means = X[:, 0, :].mean(axis=1)
chunk_stds  = X[:, 0, :].std(axis=1)
chunk_maxes = X[:, 0, :].max(axis=1)

print(f"\nPer-chunk mean  — p5: {np.percentile(chunk_means,5):.2f}  "
      f"median: {np.percentile(chunk_means,50):.2f}  "
      f"p95: {np.percentile(chunk_means,95):.2f}")
print(f"Per-chunk std   — p5: {np.percentile(chunk_stds,5):.2f}  "
      f"median: {np.percentile(chunk_stds,50):.2f}  "
      f"p95: {np.percentile(chunk_stds,95):.2f}")
print(f"Per-chunk max   — p5: {np.percentile(chunk_maxes,5):.2f}  "
      f"median: {np.percentile(chunk_maxes,50):.2f}  "
      f"p95: {np.percentile(chunk_maxes,95):.2f}")

# Flag chunks with suspiciously high or low max CO2
flat_chunks    = (chunk_maxes < 5).sum()
extreme_chunks = (chunk_maxes > 80).sum()
print(f"\nFlat chunks (max CO2 < 5 mmHg):    {flat_chunks:,}  "
      f"({flat_chunks/len(Y)*100:.1f}%) — labeled as: Safe={((chunk_maxes<5)&(Y==0)).sum():,} Crisis={((chunk_maxes<5)&(Y==1)).sum():,}")
print(f"Extreme chunks (max CO2 > 80 mmHg): {extreme_chunks:,}  "
      f"({extreme_chunks/len(Y)*100:.1f}%) — labeled as: Safe={((chunk_maxes>80)&(Y==0)).sum():,} Crisis={((chunk_maxes>80)&(Y==1)).sum():,}")

# Actual Model

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# ── Load the .npy files ──────────────────────────────────────────
X = np.load('/content/vent_X_train.npy')   # shape: [N, 1, 100]
Y = np.load('/content/vent_Y_train.npy')   # shape: [N]

print(f"X shape: {X.shape}  dtype: {X.dtype}")
print(f"Y shape: {Y.shape}  dtype: {Y.dtype}")
print(f"Class balance — Safe: {(Y==0).sum():,}  Crisis: {(Y==1).sum():,}")

# ── Custom Dataset ───────────────────────────────────────────────
class VentDataset(Dataset):
    def __init__(self, X, Y):
        # X is already float32 from the pipeline; Y needs long for BCE
        self.X = torch.from_numpy(X)                          # [N, 1, 100]
        self.Y = torch.from_numpy(Y).float()                  # [N]  float for BCEWithLogitsLoss

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]   # returns ([1, 100] tensor, scalar)

# ── Train / Validation split (80 / 20) ──────────────────────────
full_dataset = VentDataset(X, Y)
n_total     = len(full_dataset)
n_train     = int(0.8 * n_total)
n_val       = n_total - n_train

train_set, val_set = random_split(
    full_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)   # reproducible split
)

# ── DataLoaders ──────────────────────────────────────────────────
BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
import torch.nn as nn

class VentGuardian(nn.Module):
    """
    Upgraded 1D-CNN with BatchNorm and deeper feature extraction.
    Still lightweight for edge deployment — target < 1MB.
    Input:  [batch, 1, 100]
    Output: [batch]  (raw logit)
    """
    def __init__(self):
        super().__init__()

        # Block 1: catch short-term wave features (upstrokes, peaks)
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),  # second conv, same size
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)                                # 100 → 50
        )
        # Block 2: catch mid-range patterns (waveform flattening, baseline drift)
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)                                # 50 → 25
        )
        # Block 3: high-level compression
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4)                        # → 128 × 4 = 512, regardless of input size
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x.squeeze(1)


# Sanity check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VentGuardian().to(device)
dummy = torch.zeros(4, 1, 100).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")   # expect ~200-400k, still well under 1MB

In [ ]:
model = VentGuardian().to(device)

In [ ]:
import torch.optim as optim

EPOCHS    = 50          # was 20 — loss was still dropping
LR        = 5e-4

# Increase the crisis upweighting further to push precision up
# (3.11x wasn't enough — the model is still too trigger-happy on crisis)
pos_weight = torch.tensor([1.5], dtype=torch.float32).to(device)  # was n_safe/n_crisis
print(f"pos_weight: {pos_weight.item():.2f}x")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=6, factor=0.5)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_batch, Y_batch in loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            if train:
                optimizer.zero_grad()

            logits = model(X_batch)
            loss   = criterion(logits, Y_batch)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(Y_batch)
            preds  = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == Y_batch).sum().item()
            total  += len(Y_batch)

    return total_loss / total, correct / total * 100

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)

    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    marker = " ◀ best" if vl_loss < best_val_loss else ""
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), '/content/vent_guardian_best.pth')

    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2f}% | {vl_loss:>8.4f} | {vl_acc:>6.2f}%{marker}")

print("\nTraining complete.")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy (%)'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('/content/vent_guardian_best.pth'))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        logits = model(X_batch.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(Y_batch.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Sweep thresholds from 0.1 to 0.9
thresholds = np.arange(0.1, 0.91, 0.01)
f1_scores, precisions, recalls = [], [], []

for t in thresholds:
    preds = (all_probs >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    f1_scores.append(f1_score(all_labels, preds, pos_label=1, zero_division=0))
    precisions.append(precision_score(all_labels, preds, pos_label=1, zero_division=0))
    recalls.append(recall_score(all_labels, preds, pos_label=1, zero_division=0))

best_idx = np.argmax(f1_scores)
best_t   = thresholds[best_idx]
print(f"Best threshold: {best_t:.2f}")
print(f"  Crisis F1:        {f1_scores[best_idx]:.3f}")
print(f"  Crisis Precision: {precisions[best_idx]:.3f}")
print(f"  Crisis Recall:    {recalls[best_idx]:.3f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores,  label='F1')
ax.plot(thresholds, precisions, label='Precision')
ax.plot(thresholds, recalls,    label='Recall')
ax.axvline(best_t, color='red', linestyle='--', label=f'Best threshold ({best_t:.2f})')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Crisis Class: Precision / Recall / F1 vs Threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/threshold_curve.png', dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_preds = (all_probs >= best_t).astype(int)
print(f"\n--- Evaluation at threshold = {best_t:.2f} ---")
print(classification_report(all_labels, best_preds, target_names=['Safe', 'Crisis']))
print(confusion_matrix(all_labels, best_preds))

In [ ]:
torch.save(model.state_dict(), '/content/vent_guardian.pth')

# Save threshold alongside weights so edge deployment knows what to use
import json
meta = {
    'threshold': float(best_t),
    'input_shape': [1, 100],
    'classes': {0: 'Safe', 1: 'Crisis'},
    'crisis_precision': float(precisions[best_idx]),
    'crisis_recall':    float(recalls[best_idx]),
    'crisis_f1':        float(f1_scores[best_idx]),
}
with open('/content/vent_guardian_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))

import shutil
shutil.copy('/content/vent_guardian.pth',      '/content/drive/MyDrive/vent_guardian.pth')
shutil.copy('/content/vent_guardian_meta.json', '/content/drive/MyDrive/vent_guardian_meta.json')
print("\nSaved to Drive ✅")